# Phase 4C.7B — Independent Cohort Technical Pilot (Colab GPU)

> **Launcher only.** Every step calls the production CLI `scripts/research/run_cohort_acquisition.py`;
> download, preprocessing, mask generation, inpainting and QC are **not** re-implemented here.
> **Pinned code**: the repository is checked out *detached* at `EXPECTED_COMMIT` (Cell 2) and `HEAD` is asserted; a mutable branch head is never executed.

## What this notebook does
1. **Cell 1** defines fail-closed helpers (no side effects).
2. **Cell 2** mounts Google Drive (failure **stops**; there is no `/content` scratch fallback), checks out the pinned commit (a dirty clone **stops** and is left untouched), installs inference dependencies, and runs `--verify-plan` (exits non-zero unless 440 *eligible* candidates exist).
3. **Cell 3** runs the **pilot only**: exactly 2 pairs per stratum = **8 pairs**, in a new run directory `MyDrive/forensics-web-lab/phase_4c7b_runs/<RUN_ID>`. The CLI first checks the locked GPU policy (CUDA, ≥ 12 GiB VRAM; Colab T4/A100) and refuses an existing run directory unless `RESUME_RUN_ID` is set (resume re-verifies binding and SHA-256 of every artifact).
4. **Cell 4** runs only if Cell 3 completed in *this* session: it audits the run receipt against this run id, commit, protocol, catalog and plan hashes, then writes a new `<RUN_ID>_package.zip` (never overwrites).

The run records download time/rendition and provenance per authentic image, checkpoint + resolved Hugging Face revision, prompt, seed and generation parameters, SHA-256 of authentic/edited/mask files, every attempt with QC outcome and elapsed time, per-stratum wall time and peak VRAM (`torch.cuda.max_memory_allocated`), plus a contact sheet **without detector scores**. No detector is imported or run.

**Full acquisition (400 pairs) is not launched from this notebook**; it requires review of the pilot and its content QC first.

> **Current gate (see `research/evidence/phase-4c.7b/PHASE_REPORT.md`)**: the v1 catalog is superseded (0/440 candidates pass the eligibility rules). Until an eligible `verified_candidate_catalog_v2.json` is committed and this notebook is re-pinned, Cell 2's `--verify-plan` fails by design and nothing is generated.


In [ ]:
# Cell 1: Fail-closed helpers (definitions only; nothing runs here)
import subprocess
import sys
from pathlib import Path


def run(cmd, cwd=None):
    """Run a command; a non-zero exit raises CalledProcessError and stops the notebook."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=cwd, check=True)


def _git(repo_dir, *args):
    return subprocess.run(["git", *args], cwd=repo_dir, check=True, capture_output=True, text=True).stdout.strip()


def checkout_pinned(repo_dir, repo_url, sha):
    """Detached checkout of exactly `sha`. A clone with local changes is never modified."""
    repo_dir = Path(repo_dir)
    if not (repo_dir / ".git").is_dir():
        run(["git", "clone", "--no-checkout", repo_url, repo_dir])
    else:
        dirty = _git(repo_dir, "status", "--porcelain")
        if dirty:
            raise RuntimeError(f"{repo_dir} has uncommitted changes; resolve them manually (nothing was modified):\n{dirty}")
    run(["git", "fetch", "origin", sha], cwd=repo_dir)
    run(["git", "checkout", "--detach", sha], cwd=repo_dir)
    head = _git(repo_dir, "rev-parse", "HEAD")
    if head != sha:
        raise RuntimeError(f"HEAD {head} != expected {sha}")
    return head


def mount_drive_or_fail(mount_fn, mount_point):
    """Mount Google Drive or stop; acquisition outputs are never written to scratch storage."""
    try:
        mount_fn(str(mount_point))
    except Exception as e:
        raise RuntimeError(f"Google Drive mount failed ({e}); stopping without creating any run.") from e
    my_drive = Path(mount_point) / "MyDrive"
    if not my_drive.is_dir():
        raise RuntimeError(f"Google Drive mount did not expose {my_drive}; stopping without creating any run.")
    return my_drive


In [ ]:
# Cell 2: Drive, pinned checkout, dependencies, plan preflight (any failure stops here)
import datetime
from google.colab import drive

EXPECTED_COMMIT = "d487d1e79fcc626333c9f1b23e7854ef60f6a820"
REPO_URL = "https://github.com/nomozer/forensics-web-lab.git"
REPO_DIR = Path("/content/forensics-web-lab")
RESUME_RUN_ID = None  # set to an existing run id to resume it (binding + SHA-256 are re-verified)

MY_DRIVE = mount_drive_or_fail(drive.mount, "/content/drive")
RUNS_ROOT = MY_DRIVE / "forensics-web-lab" / "phase_4c7b_runs"
RUN_ID = RESUME_RUN_ID or "pilot-" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

checkout_pinned(REPO_DIR, REPO_URL, EXPECTED_COMMIT)
run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers", "accelerate", "safetensors", "huggingface_hub"])
run([sys.executable, "scripts/research/run_cohort_acquisition.py", "--verify-plan"], cwd=REPO_DIR)
print(f"Preflight PASS | commit {EXPECTED_COMMIT} | run {RUN_ID} -> {RUNS_ROOT / RUN_ID}")


In [ ]:
# Cell 3: Technical pilot via the production CLI — exactly 2 pairs per stratum (8 pairs)
cmd = [sys.executable, "scripts/research/run_cohort_acquisition.py",
       "--mode", "pilot", "--run-id", RUN_ID, "--expected-commit", EXPECTED_COMMIT,
       "--output-root", RUNS_ROOT, "--contact-sheet"]
if RESUME_RUN_ID:
    cmd.append("--resume")
run(cmd, cwd=REPO_DIR)
PILOT_RUN_COMPLETED = RUNS_ROOT / RUN_ID


In [ ]:
# Cell 4: Audit THIS session's run, then package it (never a stale run or ZIP)
import shutil

if "PILOT_RUN_COMPLETED" not in globals():
    raise RuntimeError("Pilot CLI did not complete in this session; nothing to audit or package.")
run([sys.executable, "scripts/research/run_cohort_acquisition.py", "--audit-run", PILOT_RUN_COMPLETED,
     "--mode", "pilot", "--run-id", RUN_ID, "--expected-commit", EXPECTED_COMMIT], cwd=REPO_DIR)
zip_base = RUNS_ROOT / f"{RUN_ID}_package"
if Path(f"{zip_base}.zip").exists():
    raise RuntimeError(f"{zip_base}.zip already exists; refusing to overwrite.")
shutil.make_archive(str(zip_base), "zip", PILOT_RUN_COMPLETED)
print(f"Audit PASS | package {zip_base}.zip | content QC: {PILOT_RUN_COMPLETED / 'content_qc_contact_sheet.html'}")
